# S27 · Save your model, then write the web app that serves it

You have a trained model. Today you turn it into something other people can use.
First we save the model to a file and load it back (that is the trick that lets an
app reuse your model without retraining). Then, from this very notebook, we write
the small web app that serves it and read it line by line. Finally we walk through
putting it online at a free public link.

**New here? Read this once.**

- New to Python? You can still do this whole notebook. Press the play button on
  each cell, top to bottom, and read the plain-English note above each one.
- New to the idea of "a model is a function"? The short version: numbers go in,
  an answer comes out. Any unfamiliar word is in `primers/glossary.md`.
- Never used a terminal or GitHub? That is fine. The one place we need them is
  the deploy step, and it is written out click by click.
- Already confident with code? Skip to the cells marked **Stretch (optional)**.

## How this notebook runs (important, and reassuring)

A Streamlit app is a plain Python **script** that a web server runs, not notebook
code. If we started that server inside this notebook, the notebook would hang
forever waiting for it.

So we do not start it here. This notebook only **trains, saves, and loads** the
model, then **writes** the app to a file and **prints** it so you can read it. It
never launches a server and never calls `streamlit run`. That means it runs
straight through, offline, and finishes in a couple of seconds. You run the actual
app later with a single command, shown near the end.

## Setup

If you are on **Google Colab**, run the next cell once. On your **own machine**
you already installed everything with `uv`, so the next cell does nothing there.

In [ ]:
# This notebook itself only uses numpy, scikit-learn and joblib, and Colab
# already ships all three, so there is nothing to install here.
#
# Note: the web app we write later also needs the "streamlit" library, but we do
# NOT import it in this notebook. Streamlit is only needed to RUN the app, and the
# hosting machine installs it for us from requirements.txt. So nothing to do now.
print("Setup complete - nothing to install.")

In [ ]:
import numpy as np                                  # fast maths on rows of numbers
from sklearn.datasets import load_iris                # a tiny built-in dataset
from sklearn.model_selection import train_test_split  # to hold out a fair test set
from sklearn.linear_model import LogisticRegression   # a simple, common classifier
from sklearn.metrics import accuracy_score            # to score the model honestly

## Step 1 — load a small, built-in dataset

We need some data and a model. To keep the focus on deployment, we use the tiny
**iris** flowers dataset that ships inside scikit-learn, so there is nothing to
download and it runs anywhere.

Each flower has four measurements (the inputs), and we want to guess which of three
species it is (the output). Sorting each case into one of a few named groups like
this is called **classification**. It is the same shape of problem as a bank
sorting loan applications into approve / review / reject, or a mandi sorting
mangoes into Alphonso, Kesar and Totapuri by their size and shape. Ours is just
small enough to fit in one line.

In [ ]:
# A seed so anything random is the same every time we run this.
np.random.seed(0)

# Load the dataset. X holds the measurements, y holds the species (0, 1 or 2).
iris = load_iris()
X = iris.data        # shape (150, 4): 150 flowers, 4 measurements each
y = iris.target      # shape (150,):   the species number for each flower

print("number of flowers      :", X.shape[0])
print("measurements per flower :", X.shape[1])
print("the four measurements   :", iris.feature_names)
print("the three species       :", list(iris.target_names))

## Step 2 — look at the first few rows

Always glance at your data before modelling. Each row is one flower; the four
numbers are its measurements in centimetres, and the arrow points to the true
species.

In [ ]:
# Print the measurements and the species for the first 5 flowers.
for flower_number in range(5):
    measurements = X[flower_number]
    species_number = y[flower_number]
    species_name = iris.target_names[species_number]
    print("flower", flower_number, ":", measurements, "->", species_name)

## Step 3 — split into training and test data

We train the model on one part of the data and keep another part hidden, so we can
check the model on flowers it has never seen. Judging a model only on data it
already studied would be like grading students on the exact questions they
practised. Holding out a fresh test set keeps us honest.

In [ ]:
# 80% of the flowers for training, 20% kept aside for a fair test.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0)

print("training flowers:", X_train.shape[0])
print("test flowers    :", X_test.shape[0])

## Step 4 — train a tiny model

We use **logistic regression**, a simple and very common classifier. Calling
`.fit(...)` is the training step: the model looks at the training flowers and
learns the rule that maps measurements to a species. This is the one line that
turns raw data into the function `f` we will serve.

In [ ]:
# Create the model. max_iter just gives it enough steps to settle down.
model = LogisticRegression(max_iter=200)

# Train it on the TRAINING data only.
model.fit(X_train, y_train)

print("The model is trained.")

## Step 5 — check it on the hidden test flowers

Accuracy is the share of test flowers the model labels correctly. We look at it so
we know the model is actually worth saving and shipping.

In [ ]:
# Ask the trained model to predict the species of the hidden test flowers.
test_predictions = model.predict(X_test)

# Compare the predictions with the true answers.
accuracy = accuracy_score(y_test, test_predictions)
print("accuracy on the hidden test flowers:", round(accuracy, 3))

## Step 6 — save the trained model to a file (joblib)

This is the key step, and it is the whole reason an app can be fast. The trained
`model` lives in memory right now, so it vanishes when the notebook closes.
`joblib.dump` writes it to a file on disk. Think of it as freezing the function
`f` so we can thaw it later, in another program, on another day, with no
retraining. Writing an object to a file like this is called **serializing** it.

In [ ]:
import joblib
import os

# The file we will write the model into. ".joblib" is just a sensible name;
# you will often also see ".pkl".
model_filename = "iris_model.joblib"

# Save (serialise) the trained model object to that file. This is ONE line.
joblib.dump(model, model_filename)

print("Saved the trained model to:", model_filename)

# Show that the file really exists on disk now, and how small it is.
file_size_in_bytes = os.path.getsize(model_filename)
print("file size:", file_size_in_bytes, "bytes")

## Step 7 — load the model back from the file

Now we pretend we are a brand-new program, for example the web app. We do **not**
retrain anything. We just read the file back into a fresh variable with
`joblib.load`. That fresh variable is the same trained function.

In [ ]:
# Load (read back) the model from the file into a new variable.
loaded_model = joblib.load(model_filename)

print("Loaded the model back from:", model_filename)
print("type of the loaded object:", type(loaded_model).__name__)

## Step 8 — predict with the LOADED model

Finally, we use the model we just loaded to guess the species of one new flower.
This is exactly what the web app does on every click: load the saved function
once, then call it on whatever the user typed.

In [ ]:
# One new flower's four measurements (sepal length, sepal width,
# petal length, petal width), in centimetres. The model expects a 2D array with
# one row per flower, so we wrap the four numbers in another list.
new_flower = [[5.1, 3.5, 1.4, 0.2]]

# Use the LOADED model (not the original one) to predict.
predicted_species_number = loaded_model.predict(new_flower)[0]
predicted_species_name = iris.target_names[predicted_species_number]

print("measurements of the new flower:", new_flower[0])
print("predicted species number      :", predicted_species_number)
print("predicted species name        :", predicted_species_name)

## Step 9 — confirm the loaded model agrees with the original

Saving and loading must not change the model's behaviour. We check that the loaded
model gives the exact same predictions as the original one on the test flowers.

In [ ]:
# Predictions from the original model and from the loaded model.
original_predictions = model.predict(X_test)
loaded_predictions = loaded_model.predict(X_test)

# np.array_equal is True only if every prediction matches.
they_match = np.array_equal(original_predictions, loaded_predictions)

print("loaded model matches the original on every test flower:", they_match)

### Stretch (optional) — show how confident the model is

Skip this if you are new to code. A good app does not just say "setosa"; it says
how sure it is. Most scikit-learn classifiers can also return a **probability** for
each class with `predict_proba`. The app uses exactly this to show a confidence
percentage next to the answer.

In [ ]:
# Probability the model gives to each of the three species, for our new flower.
probabilities = loaded_model.predict_proba(new_flower)[0]

for species_number in range(len(probabilities)):
    species_name = iris.target_names[species_number]
    percent = round(100 * probabilities[species_number], 1)
    print(species_name, ":", percent, "%")

# The confidence for the species the model actually picked.
chosen_confidence = round(100 * probabilities[predicted_species_number], 1)
print()
print("The model picked", predicted_species_name,
      "with", chosen_confidence, "% confidence.")

## Step 10 — now write the web app that serves this model

Here is the pivot from "a model in a notebook" to "a model on the internet". The
web app is a plain Python script called `app.py`. We write it to disk from here,
so this notebook produces the exact file you will deploy. You do not have to
copy or retype anything.

Read the reminder in the cell: we only **write** the app. We do not start it. That
is on purpose, so this notebook keeps running instead of hanging on a server.

In [ ]:
# The full text of our Streamlit web app, kept as one block of text.
# We write it straight to app.py so this notebook produces the exact file
# you deploy - nothing to copy by hand, nothing to mistype.
#
# We only WRITE the file here. We do NOT run it. Starting a web server inside
# a notebook would hang forever; you run the app later with one command.
app_source = r'''# app.py - a tiny Streamlit web app that serves a trained model.
#
# Run it locally from this folder with:
#     streamlit run app.py
#
# Streamlit's mental model: this whole script runs again, top to bottom, every
# time the user changes an input. So we read the script like a recipe.
#
# IMPORTANT: this is a plain Python SCRIPT, not a notebook. Do not run it cell
# by cell, and do not run it inside a notebook. Run it with the
# "streamlit run app.py" command shown above. (The session notebook also writes
# this exact file to disk for you, so you never have to copy it by hand.)

import streamlit as st          # the web-app library (the only new tool here)
import numpy as np              # to build the input row of numbers
import joblib                   # to load the model file we saved in the notebook
from sklearn.datasets import load_iris   # only for the species names below


# --- Block 1: load the trained model (once) -------------------------------- #
# @st.cache_resource tells Streamlit: run this function only ONCE, then reuse
# its result on every later rerun. Without it, the file would be re-read on
# every click, which is slow and wasteful.
@st.cache_resource
def load_model():
    # Read the model we saved in the notebook with joblib.dump.
    # The file sits next to this script in the repo, so a plain name works.
    return joblib.load("iris_model.joblib")


# Load the model and the human-readable species names.
model = load_model()
species_names = load_iris().target_names   # e.g. setosa, versicolor, virginica


# --- Block 2: the page heading --------------------------------------------- #
st.title("Iris flower classifier")
st.write(
    "Enter a flower's four measurements (in centimetres) and the model will "
    "guess its species. The model was trained once and saved; this app just "
    "loads it and predicts."
)


# --- Block 3: collect the inputs from the user ----------------------------- #
# Each slider is one function call that returns the number the user chose.
# We put the sliders in the sidebar to keep the page tidy.
st.sidebar.header("Flower measurements")
sepal_length = st.sidebar.slider("Sepal length (cm)", 4.0, 8.0, 5.1)
sepal_width = st.sidebar.slider("Sepal width (cm)", 2.0, 4.5, 3.5)
petal_length = st.sidebar.slider("Petal length (cm)", 1.0, 7.0, 1.4)
petal_width = st.sidebar.slider("Petal width (cm)", 0.1, 2.5, 0.2)

# Echo the inputs back so the user can see exactly what was used.
st.subheader("Your inputs")
st.write("Sepal length:", sepal_length, "cm")
st.write("Sepal width :", sepal_width, "cm")
st.write("Petal length:", petal_length, "cm")
st.write("Petal width :", petal_width, "cm")


# --- Block 4: predict and display the answer ------------------------------- #
# The model expects a 2D array with one row per flower, so we make one row.
# The order of the numbers MUST match the order used during training.
input_row = np.array([[sepal_length, sepal_width, petal_length, petal_width]])

# Call the model: this is the f(x) step, the whole point of the app.
predicted_number = model.predict(input_row)[0]
predicted_name = species_names[predicted_number]

# How confident the model is, as a percentage for the chosen species.
probabilities = model.predict_proba(input_row)[0]
confidence_percent = round(100 * probabilities[predicted_number], 1)

# Show the prediction big and clear, with its confidence as context.
st.subheader("Prediction")
st.metric(label="Predicted species", value=predicted_name)
st.write("Model confidence:", confidence_percent, "%")
'''

with open("app.py", "w") as app_file:
    app_file.write(app_source)

print("Wrote app.py to this folder -", len(app_source),
      "characters of plain Python.")

## Step 11 — read the app you just wrote

Let us print `app.py` and read it. It is about sixty lines, and it is the same
four blocks every model app has. Do not worry about memorising Streamlit; just see
the shape.

In [ ]:
# Show the app we just wrote, so you can read it without leaving the notebook.
with open("app.py") as app_file:
    print(app_file.read())

## The four blocks of every model app

Read the printed script above and you will find these four parts, in order.

1. **Load the model** — one cached call near the top reads `iris_model.joblib`
   once and keeps it in memory. That caching (`st.cache_resource`) is why the app
   does not reload the file on every click.
2. **Collect the inputs** — a slider for each measurement. Every widget, like
   `st.slider(...)`, is one function call that returns the number the user chose.
3. **Predict** — the four numbers are put into one row and passed to
   `model.predict(...)`. That single line is the `f(x)` call, the whole point.
4. **Display** — the answer is shown big and clear, with a confidence percentage
   for context, never a bare number.

That is it. The hard part, training the model, you did weeks ago. The app is just
these four blocks of plumbing around it.

## Step 12 — write requirements.txt (what the cloud must install)

When you deploy, the hosting machine starts empty. It has none of your libraries.
`requirements.txt` is the shopping list it reads to install exactly the right
packages, at exactly the right versions. Pinning versions matters: a model saved
with one scikit-learn version may not load on a different one.

In [ ]:
# Write the requirements file next to the app, so your deploy folder is
# complete. The cloud reads this and installs each package at these versions.
requirements_text = '''# Packages the Streamlit app needs. The cloud machine starts empty and installs
# exactly these. Pinning versions keeps the cloud matching your laptop, so the
# saved model loads cleanly.
streamlit==1.40.1
scikit-learn==1.8.0
numpy==2.4.6
joblib==1.5.3
'''

with open("requirements.txt", "w") as req_file:
    req_file.write(requirements_text)

print("Wrote requirements.txt:")
print(requirements_text)

## Step 13 — how to actually run the app (in a terminal, not here)

You now have three files in this folder: `iris_model.joblib`, `app.py` and
`requirements.txt`. To see the app on your own machine, open a **terminal** in
this folder and run this one command:

```
streamlit run app.py
```

Streamlit starts a small web server and opens the app in your browser, usually at
`http://localhost:8501`. Move the sliders and watch the prediction update. Press
`Ctrl+C` in the terminal to stop it.

Do not run `streamlit run` inside a notebook cell. It starts a server that never
finishes, so the cell would spin forever. The terminal is its home.

## Step 14 — deploy it free to a public URL

This is the payoff: a link anyone can open, on any phone, for free. Both options
below have a genuinely free tier with no credit card required. Pick one.

**Option A — Streamlit Community Cloud**

1. Put `app.py`, `iris_model.joblib`, `requirements.txt` and a short `README.md`
   in a **public GitHub repo** (the same Git workflow from S4).
2. Go to [share.streamlit.io](https://share.streamlit.io), sign in with GitHub,
   and click **New app**.
3. Choose your repo and point it at `app.py`.
4. Streamlit reads `requirements.txt`, installs the packages, runs the app, and
   gives you a public URL like `https://your-app.streamlit.app`.
5. Put that live URL in your README. Anyone, anywhere, can now open the link.

**Option B — Hugging Face Spaces** (equally free)

1. Create a new **Space** at [huggingface.co/spaces](https://huggingface.co/spaces)
   and choose **Streamlit** as the SDK.
2. Push the same files to the Space's repo.
3. The Space builds from `requirements.txt` and gives you a public URL.

Updating the app later is just a `git push`; the cloud redeploys automatically.

A few things to watch:

- **Pin versions** in `requirements.txt` (already done for you). A model saved
  with one scikit-learn version may fail to load on another.
- **Use relative paths** inside the repo (`joblib.load("iris_model.joblib")`),
  never a laptop path like `C:/Users/me/...`. A hardcoded laptop path works on
  your machine and dies in the cloud.
- **Never commit secrets or API keys.** Use the platform's secrets manager.
- Free apps **sleep when idle**; the first visit after a nap takes 10 to 30
  seconds to wake. That is fine for a demo, just warn whoever you send the link
  to.

This is demo-grade deployment: perfect for a portfolio, a recruiter, or your
capstone, but not production infrastructure (no scaling, monitoring or SLAs). We
keep deliberately to Streamlit and its free hosting: no Docker, no FastAPI.

## What you just did

You trained a model, **saved it to a file**, and **loaded it back** to predict,
with no retraining. Then, from this notebook, you **wrote the web app** that serves
it and read its four blocks, and you have the exact steps to put it online at a
free public link.

Notice what this notebook did and did not do. It produced three real files in this
folder (`iris_model.joblib`, `app.py`, `requirements.txt`), and it ran straight
through without ever starting a server. That is the point: the notebook builds the
app; the terminal (or the free cloud) runs it.

That saved file is a frozen function `f(x)`, and a web app is nothing more than
that function with a public address. That is what "a model as a function exposed
as a service" means.